In [ ]:
# EXERCISE 7: BOLTZMANN AND CAUCHY MACHINES
# Stochastic Feature Selection and Energy Optimization
# Dataset: Wine Quality Dataset
# Run this entire code in ONE Google Colab cell.

import numpy as np
import pandas as pd
import warnings
import time
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")
np.random.seed(42)

# -------------------- LOAD DATASET --------------------

def load_wine_data():
    urls = [
        "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv",
        "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"
    ]

    frames = []

    for url in urls:
        try:
            data = pd.read_csv(url, sep=";")
            frames.append(data)
        except Exception as e:
            print("Could not load:", url)

    if not frames:
        raise RuntimeError(
            "Dataset download failed. Check your internet connection and rerun."
        )

    # Combine red and white wine records.
    df = pd.concat(frames, ignore_index=True)

    # Binary classification: quality >= 6 is good, otherwise not good.
    X_df = df.drop(columns=["quality"])
    y = (df["quality"] >= 6).astype(int).values

    feature_names = list(X_df.columns)
    X = X_df.values.astype(float)

    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    return X_train, X_test, y_train, y_test, feature_names


print("Loading Wine Quality Dataset...")
X_train, X_test, y_train, y_test, FEATURES = load_wine_data()
N_FEATURES = len(FEATURES)

print("Dataset loaded successfully!")
print("Number of features:", N_FEATURES)
print("Features:", FEATURES)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Target: 0 = Quality < 6, 1 = Quality >= 6")


# -------------------- CLASSIFICATION AND ENERGY --------------------

# Cache evaluation results so repeated feature states are evaluated only once.
evaluation_cache = {}

def evaluate_state(state):
    state = np.asarray(state, dtype=int)
    key = tuple(state.tolist())

    if key in evaluation_cache:
        return evaluation_cache[key]

    selected = np.where(state == 1)[0]

    # Require at least one selected feature.
    if len(selected) == 0:
        result = (0.0, 1.0)
        evaluation_cache[key] = result
        return result

    clf = RandomForestClassifier(
        n_estimators=40,
        max_depth=8,
        random_state=42,
        n_jobs=-1
    )

    clf.fit(X_train[:, selected], y_train)
    predictions = clf.predict(X_test[:, selected])
    accuracy = accuracy_score(y_test, predictions)

    # Energy = classification error + feature-count penalty.
    error = 1.0 - accuracy
    feature_penalty = 0.10 * len(selected) / N_FEATURES
    energy = error + feature_penalty

    result = (accuracy, energy)
    evaluation_cache[key] = result
    return result


def initial_state():
    # Random binary vector, ensuring at least one selected feature.
    state = np.random.randint(0, 2, N_FEATURES)
    if np.sum(state) == 0:
        state[np.random.randint(N_FEATURES)] = 1
    return state


def selected_names(state):
    return [FEATURES[i] for i in range(N_FEATURES) if state[i] == 1]


def display_state(state):
    accuracy, energy = evaluate_state(state)
    print("State:", state.tolist())
    print("Selected Features:", selected_names(state))
    print("Selected Feature Count:", int(np.sum(state)))
    print(f"Classification Accuracy: {accuracy * 100:.2f}%")
    print(f"Energy: {energy:.6f}")


# -------------------- STOCHASTIC MACHINES --------------------

def boltzmann_search(start_state, iterations=60, initial_temp=0.10,
                     cooling=0.95, patience=12):
    rng = np.random.default_rng()
    current = np.asarray(start_state, dtype=int).copy()
    current_accuracy, current_energy = evaluate_state(current)

    best = current.copy()
    best_energy = current_energy
    best_accuracy = current_accuracy

    temperature = initial_temp
    accepted_states = [current.copy()]
    transition_log = []
    seen_states = [tuple(current.tolist())]
    higher_accepted = 0
    reductions = 0
    escaped = 0
    local_minima = 0
    stagnant = 0
    trapped_iterations = 0
    escape_iteration = None
    local_min_energy = None

    for iteration in range(1, iterations + 1):
        candidate = current.copy()
        index = rng.integers(0, N_FEATURES)
        candidate[index] = 1 - candidate[index]

        # Prevent empty feature subsets.
        if np.sum(candidate) == 0:
            candidate[index] = 1

        candidate_accuracy, candidate_energy = evaluate_state(candidate)
        delta = candidate_energy - current_energy

        probability = (
            1.0 if delta <= 0
            else np.exp(-delta / max(temperature, 1e-10))
        )
        accepted = rng.random() < probability

        if accepted:
            current = candidate
            current_accuracy = candidate_accuracy
            current_energy = candidate_energy
            accepted_states.append(current.copy())
            seen_states.append(tuple(current.tolist()))

            if delta < -1e-12:
                reductions += 1
                stagnant = 0
            else:
                stagnant += 1

            if delta > 1e-12:
                higher_accepted += 1
        else:
            stagnant += 1

        if current_energy < best_energy:
            best = current.copy()
            best_energy = current_energy
            best_accuracy = current_accuracy

        if stagnant >= patience:
            local_minima += 1
            if local_min_energy is None:
                local_min_energy = current_energy
            trapped_iterations += stagnant

            # Continue stochastic exploration; temperature is reheated.
            temperature = min(temperature * 1.8, initial_temp)
            stagnant = 0

            if current_energy < local_min_energy - 1e-12:
                escaped += 1
                if escape_iteration is None:
                    escape_iteration = iteration

        transition_log.append({
            "Iteration": iteration,
            "Feature Changed": FEATURES[index],
            "Energy Change": delta,
            "Acceptance Probability": probability,
            "Accepted": accepted,
            "Current Energy": current_energy,
            "Temperature": temperature
        })

        temperature *= cooling

    return {
        "machine": "Boltzmann",
        "initial_state": np.asarray(start_state, dtype=int).copy(),
        "initial_energy": evaluate_state(start_state)[1],
        "best_state": best,
        "best_energy": best_energy,
        "best_accuracy": best_accuracy,
        "final_energy": current_energy,
        "accepted_states": accepted_states,
        "transitions": transition_log,
        "unique_states": len(set(seen_states)),
        "repeated_states": len(seen_states) - len(set(seen_states)),
        "higher_accepted": higher_accepted,
        "reductions": reductions,
        "local_minima": local_minima,
        "local_minima_escaped": escaped,
        "trapped_iterations": trapped_iterations,
        "escape_iteration": escape_iteration,
        "iterations": iterations
    }


def cauchy_search(start_state, iterations=60, initial_temp=0.10,
                  cooling=0.95, patience=12):
    rng = np.random.default_rng()
    current = np.asarray(start_state, dtype=int).copy()
    current_accuracy, current_energy = evaluate_state(current)

    best = current.copy()
    best_energy = current_energy
    best_accuracy = current_accuracy

    temperature = initial_temp
    accepted_states = [current.copy()]
    transition_log = []
    seen_states = [tuple(current.tolist())]
    higher_accepted = 0
    reductions = 0
    escaped = 0
    local_minima = 0
    stagnant = 0
    trapped_iterations = 0
    escape_iteration = None
    local_min_energy = None

    for iteration in range(1, iterations + 1):
        candidate = current.copy()

        # Cauchy-distributed perturbation controls the number of flipped bits.
        perturbation = rng.standard_cauchy() * max(temperature, 0.05)
        flips = max(1, min(N_FEATURES, int(abs(perturbation) * N_FEATURES) + 1))
        indices = rng.choice(N_FEATURES, size=flips, replace=False)
        candidate[indices] = 1 - candidate[indices]

        if np.sum(candidate) == 0:
            candidate[rng.integers(N_FEATURES)] = 1

        candidate_accuracy, candidate_energy = evaluate_state(candidate)
        delta = candidate_energy - current_energy

        # Cauchy-inspired acceptance gives occasional uphill moves.
        probability = (
            1.0 if delta <= 0
            else 1.0 / (1.0 + (delta / max(temperature, 1e-10)) ** 2)
        )
        accepted = rng.random() < probability

        if accepted:
            current = candidate
            current_accuracy = candidate_accuracy
            current_energy = candidate_energy
            accepted_states.append(current.copy())
            seen_states.append(tuple(current.tolist()))

            if delta < -1e-12:
                reductions += 1
                stagnant = 0
            else:
                stagnant += 1

            if delta > 1e-12:
                higher_accepted += 1
        else:
            stagnant += 1

        if current_energy < best_energy:
            best = current.copy()
            best_energy = current_energy
            best_accuracy = current_accuracy

        if stagnant >= patience:
            local_minima += 1
            if local_min_energy is None:
                local_min_energy = current_energy
            trapped_iterations += stagnant

            # Cauchy machine naturally allows larger perturbations.
            temperature = min(temperature * 1.8, initial_temp)
            stagnant = 0

            if current_energy < local_min_energy - 1e-12:
                escaped += 1
                if escape_iteration is None:
                    escape_iteration = iteration

        transition_log.append({
            "Iteration": iteration,
            "Cauchy Perturbation": perturbation,
            "Features Flipped": flips,
            "Energy Change": delta,
            "Acceptance Probability": probability,
            "Accepted": accepted,
            "Current Energy": current_energy,
            "Temperature": temperature
        })

        temperature *= cooling

    return {
        "machine": "Cauchy",
        "initial_state": np.asarray(start_state, dtype=int).copy(),
        "initial_energy": evaluate_state(start_state)[1],
        "best_state": best,
        "best_energy": best_energy,
        "best_accuracy": best_accuracy,
        "final_energy": current_energy,
        "accepted_states": accepted_states,
        "transitions": transition_log,
        "unique_states": len(set(seen_states)),
        "repeated_states": len(seen_states) - len(set(seen_states)),
        "higher_accepted": higher_accepted,
        "reductions": reductions,
        "local_minima": local_minima,
        "local_minima_escaped": escaped,
        "trapped_iterations": trapped_iterations,
        "escape_iteration": escape_iteration,
        "iterations": iterations
    }


# -------------------- MENU 1 --------------------

def generate_initial_state():
    print("\n--- GENERATE INITIAL FEATURE STATE ---")
    state = initial_state()
    display_state(state)


# -------------------- MENU 2 --------------------

def boltzmann_feature_selection():
    print("\n--- BOLTZMANN MACHINE FEATURE SELECTION ---")
    state = initial_state()
    result = boltzmann_search(state, iterations=60)

    print("\nBest Feature Subset:")
    display_state(result["best_state"])

    print("\nLast 10 State Transitions:")
    print(pd.DataFrame(result["transitions"]).tail(10).to_string(index=False))


# -------------------- MENU 3 --------------------

def cauchy_feature_selection():
    print("\n--- CAUCHY MACHINE FEATURE SELECTION ---")
    state = initial_state()
    result = cauchy_search(state, iterations=60)

    print("\nBest Feature Subset:")
    display_state(result["best_state"])

    print("\nLast 10 State Transitions:")
    print(pd.DataFrame(result["transitions"]).tail(10).to_string(index=False))


# -------------------- MENU 4 --------------------

def transition_analyzer():
    print("\n--- ENERGY STATE TRANSITION ANALYZER ---")
    state = initial_state()

    results = [
        boltzmann_search(state, iterations=60),
        cauchy_search(state, iterations=60)
    ]

    for result in results:
        print(f"\n{result['machine']} Machine")
        print("Energy Reductions:", result["reductions"])
        print("Higher-Energy States Accepted:", result["higher_accepted"])
        print("Repeated States:", result["repeated_states"])
        print("Unique States Explored:", result["unique_states"])
        print(f"Lowest Energy: {result['best_energy']:.6f}")

        print("\nAccepted-State Energy Sequence:")
        sequence = [
            evaluate_state(s)[1] for s in result["accepted_states"]
        ]
        print(np.round(sequence, 5).tolist())


# -------------------- MENU 5 --------------------

def local_minimum_escape():
    print("\n--- LOCAL MINIMUM ESCAPE CHALLENGE ---")
    state = initial_state()

    print("Both machines start from the same state:")
    display_state(state)

    b = boltzmann_search(state, iterations=100, patience=8)
    c = cauchy_search(state, iterations=100, patience=8)

    for result in [b, c]:
        print(f"\n{result['machine']} Machine")
        print(f"Best Energy: {result['best_energy']:.6f}")
        print("Local Minimum Events:", result["local_minima"])
        print("Trapped Iterations:", result["trapped_iterations"])
        print("Escape Iteration:", result["escape_iteration"])
        print("Local Minima Escaped:", result["local_minima_escaped"])

    escapes = [
        (r["escape_iteration"], r["machine"])
        for r in [b, c]
        if r["escape_iteration"] is not None
    ]

    if escapes:
        escapes.sort(key=lambda item: item[0])
        print(f"\nFirst Recorded Escape: {escapes[0][1]} Machine "
              f"at iteration {escapes[0][0]}")
    else:
        print("\nNeither machine recorded an escape under the defined criterion.")


# -------------------- MENU 6 --------------------

def feature_stability():
    print("\n--- FEATURE STABILITY ANALYZER: 20 RUNS PER MACHINE ---")

    frequencies = {
        "Boltzmann": np.zeros(N_FEATURES, dtype=int),
        "Cauchy": np.zeros(N_FEATURES, dtype=int)
    }

    for run in range(20):
        state = initial_state()

        b = boltzmann_search(state, iterations=40)
        c = cauchy_search(state, iterations=40)

        for i in range(N_FEATURES):
            frequencies["Boltzmann"][i] += b["best_state"][i]
            frequencies["Cauchy"][i] += c["best_state"][i]

        print(f"Completed run {run + 1}/20")

    for machine, counts in frequencies.items():
        print(f"\n{machine} Feature Selection Frequency")
        rows = []

        for i, feature in enumerate(FEATURES):
            frequency = counts[i]
            percentage = frequency / 20 * 100

            if percentage >= 70:
                category = "Highly Stable"
            elif percentage >= 30:
                category = "Moderately Stable"
            else:
                category = "Rarely Selected"

            rows.append({
                "Feature": feature,
                "Selected Runs": int(frequency),
                "Frequency (%)": percentage,
                "Category": category
            })

        report = pd.DataFrame(rows).sort_values(
            "Frequency (%)", ascending=False
        )
        print(report.to_string(index=False))


# -------------------- MENU 7 --------------------

def optimization_battle():
    print("\n--- BOLTZMANN VS CAUCHY OPTIMIZATION BATTLE ---")
    state = initial_state()

    print("Common Initial State:", state.tolist())

    b = boltzmann_search(state, iterations=80)
    c = cauchy_search(state, iterations=80)

    rows = []

    for result in [b, c]:
        rows.append({
            "Machine": result["machine"],
            "Initial Energy": result["initial_energy"],
            "Minimum Energy": result["best_energy"],
            "Unique States": result["unique_states"],
            "Local Minima Escaped": result["local_minima_escaped"],
            "Higher-Energy Accepted": result["higher_accepted"],
            "Best Accuracy (%)": result["best_accuracy"] * 100
        })

    report = pd.DataFrame(rows)
    print(report.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    winner = min([b, c], key=lambda r: r["best_energy"])
    print(f"\nBest Result by Minimum Energy: {winner['machine']} Machine")
    print(f"Best Energy: {winner['best_energy']:.6f}")
    print(f"Best Accuracy: {winner['best_accuracy'] * 100:.2f}%")


# -------------------- MENU 8 --------------------

def best_feature_subset_finder():
    print("\n--- BEST FEATURE SUBSET FINDER ---")

    candidates = []

    for run in range(5):
        state = initial_state()
        print(f"\nStarting state {run + 1}: {state.tolist()}")

        b = boltzmann_search(state, iterations=60)
        c = cauchy_search(state, iterations=60)

        candidates.extend([b, c])

        for result in [b, c]:
            print(f"{result['machine']} minimum energy: "
                  f"{result['best_energy']:.6f}")

    best = min(candidates, key=lambda r: r["best_energy"])
    chosen = set(selected_names(best["best_state"]))
    removed = [feature for feature in FEATURES if feature not in chosen]

    print("\nOVERALL BEST FEATURE SUBSET")
    print("Machine:", best["machine"])
    print("Selected Features:", sorted(chosen))
    print("Removed Features:", removed)
    print("Total Features:", N_FEATURES)
    print("Selected Feature Count:", len(chosen))
    print(f"Classification Accuracy: {best['best_accuracy'] * 100:.2f}%")
    print(f"Energy: {best['best_energy']:.6f}")


# -------------------- MENU 9 --------------------

def stochastic_report():
    print("\n--- STOCHASTIC OPTIMIZATION REPORT ---")
    state = initial_state()

    results = [
        boltzmann_search(state, iterations=60),
        cauchy_search(state, iterations=60)
    ]

    rows = []

    for result in results:
        rows.append({
            "Machine": result["machine"],
            "Initial Energy": result["initial_energy"],
            "Final Energy": result["best_energy"],
            "Selected Features": int(np.sum(result["best_state"])),
            "Accuracy (%)": result["best_accuracy"] * 100,
            "Unique States": result["unique_states"],
            "Local Minima": result["local_minima"],
            "Local Minima Escaped": result["local_minima_escaped"],
            "Iterations": result["iterations"]
        })

    report = pd.DataFrame(rows)
    print(report.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    best = min(results, key=lambda r: r["best_energy"])
    print("\nFINAL CONCLUSION")
    print("Best Machine:", best["machine"])
    print("Best Feature Subset:", selected_names(best["best_state"]))
    print(f"Best Energy: {best['best_energy']:.6f}")
    print(f"Classification Accuracy: {best['best_accuracy'] * 100:.2f}%")


# -------------------- MAIN MENU --------------------

def main():
    while True:
        print("\n" + "=" * 65)
        print(" WINE QUALITY: BOLTZMANN & CAUCHY FEATURE OPTIMIZATION")
        print("=" * 65)
        print("1. Generate Initial Feature State")
        print("2. Boltzmann Machine Feature Selection")
        print("3. Cauchy Machine Feature Selection")
        print("4. Energy State Transition Analyzer")
        print("5. Local Minimum Escape Challenge")
        print("6. Feature Stability Analyzer")
        print("7. Boltzmann vs Cauchy Optimization Battle")
        print("8. Best Feature Subset Finder")
        print("9. Stochastic Optimization Report")
        print("10. Exit")

        choice = input("\nEnter your choice (1-10): ").strip()

        try:
            if choice == "1":
                generate_initial_state()
            elif choice == "2":
                boltzmann_feature_selection()
            elif choice == "3":
                cauchy_feature_selection()
            elif choice == "4":
                transition_analyzer()
            elif choice == "5":
                local_minimum_escape()
            elif choice == "6":
                feature_stability()
            elif choice == "7":
                optimization_battle()
            elif choice == "8":
                best_feature_subset_finder()
            elif choice == "9":
                stochastic_report()
            elif choice == "10":
                print("Exiting Stochastic Feature Optimization System.")
                break
            else:
                print("Invalid choice. Enter a number from 1 to 10.")

        except Exception as error:
            print("Error:", error)


main()

Loading Wine Quality Dataset...
Dataset loaded successfully!
Number of features: 11
Features: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']
Training samples: 4872
Testing samples: 1625
Target: 0 = Quality < 6, 1 = Quality >= 6

 WINE QUALITY: BOLTZMANN & CAUCHY FEATURE OPTIMIZATION
1. Generate Initial Feature State
2. Boltzmann Machine Feature Selection
3. Cauchy Machine Feature Selection
4. Energy State Transition Analyzer
5. Local Minimum Escape Challenge
6. Feature Stability Analyzer
7. Boltzmann vs Cauchy Optimization Battle
8. Best Feature Subset Finder
9. Stochastic Optimization Report
10. Exit
